In [53]:
import pandas as pd

In [54]:
census_for_classify_file_list = ['Tenure - households', 'Ethnic group', 'Occupation', 'Five year age bands', 'Household composition']

In [55]:
# Directory where datasets are stored
data_dir = "../Dataset/Census 2011-2021/"

key_columns = ["local authority code", "local authority name", "LSOA code"]

# Dict to store successfully loaded census data
census_data = {}

for census_file in census_for_classify_file_list:

    file_path = f"{data_dir}{census_file}.xlsx"
    
    success = True
    
    try:
        # Dictionary mapping sheet names to suffixes
        sheets_with_suffix = {
            "2011": "_2011",
            "2021": "_2021",
        }

        # Read Excel sheets into a dictionary of DataFrames
        data_dict = {name: pd.read_excel(file_path, sheet_name=name).dropna(axis=1, how="all") for name in sheets_with_suffix}

        # Rename columns by appending the corresponding suffix to non-key columns
        for name, suffix in sheets_with_suffix.items():
            data_dict[name].rename(columns=lambda x: x + suffix if x not in key_columns else x, inplace=True)

        # Start with the first dataframe
        merged_df = data_dict[list(data_dict.keys())[0]]

        # Merge all successfully loaded datasets based on the key columns
        for name in sheets_with_suffix.keys():
            if name != list(data_dict.keys())[0]:  # Skip the first dataframe as it's already included
                merged_df = pd.merge(merged_df, data_dict[name], on=key_columns, how='outer')

    except Exception as e:
        # Handle errors
        print(f"Error: Failed to load {census_file} - {e}")
        success = False

    # Store data if successfully loaded
    if success:
        census_data[census_file] = merged_df
        print(f"Successfully loaded: {census_file}")


df_census = census_data[list(census_data.keys())[0]]

# Merge all successfully loaded datasets
for name in census_data.keys():
    if name != list(census_data.keys())[0]:  # Skip the first dataframe as it's already included
        df_census = pd.merge(df_census, census_data[name], on=key_columns, how='outer')

df_census  

Successfully loaded: Tenure - households
Successfully loaded: Ethnic group
Successfully loaded: Occupation
Successfully loaded: Five year age bands
Successfully loaded: Household composition


,LSOA code,local authority code,local authority name,All Households_2011,Owned outright_2011,Owned with a mortgage or loan_2011,Shared ownership _2011,Rented from Local Authority_2011,Other social rented_2011,Private landlord or letting agency_2011,...,Married or civil partnership couple: No children_2021,Married or civil partnership couple: Dependent children_2021,Married or civil partnership couple: non-dependent children_2021,Cohabiting couple: No children_2021,Cohabiting couple: Dependent children_2021,Cohabiting couple: Non-dependent children_2021,Lone parent: dependent children_2021,Lone parent: non-dependent children_2021,Other with dependent children_2021,All other types_2021
0,E01000001,E09000001,City of London,876,355,178,3,33,8,247,...,112,68,13,99,13,0,10,7,1,61
1,E01000002,E09000001,City of London,830,314,213,8,44,4,206,...,105,44,10,96,10,3,13,15,2,64
2,E01000003,E09000001,City of London,817,184,143,1,239,56,166,...,108,44,15,104,16,5,22,21,1,73
3,E01000005,E09000001,City of London,467,24,22,0,133,179,92,...,26,46,16,28,12,4,29,25,20,47
4,E01032739,E09000001,City of London,676,104,88,1,5,6,358,...,92,27,10,148,1,0,7,8,4,154
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4989,E01035718,E09000033,Westminster,996,380,122,4,7,37,328,...,102,87,34,27,4,3,23,29,41,49
4990,E01035719,E09000033,Westminster,567,142,100,8,95,26,161,...,52,69,18,48,12,1,38,25,12,67
4991,E01035720,E09000033,Westminster,558,78,91,14,107,104,134,...,37,52,10,52,23,1,33,54,5,74
4992,E01035721,E09000033,Westminster,1482,195,117,20,279,454,365,...,99,89,30,49,8,4,64,59,27,81


In [56]:
df_census.drop(columns=['All usual residents_2011_y', 'All usual residents_2021_y', 'All households_2011', 'All households_2021'], inplace=True)
df_census.rename(columns={'All usual residents_2011_x':'All usual residents_2011', 'All usual residents_2021_x':'All usual residents_2021'}, inplace=True)

In [57]:
df_property_sales = pd.read_csv("../Output/Number of sales mapping.csv")

In [58]:
df_census = df_census.merge(df_property_sales, left_on='LSOA code', right_on='LSOA21CD')
df_census.drop(columns=['LSOA21CD'], inplace=True)

In [59]:
df_census

,LSOA code,local authority code,local authority name,All Households_2011,Owned outright_2011,Owned with a mortgage or loan_2011,Shared ownership _2011,Rented from Local Authority_2011,Other social rented_2011,Private landlord or letting agency_2011,...,Married or civil partnership couple: non-dependent children_2021,Cohabiting couple: No children_2021,Cohabiting couple: Dependent children_2021,Cohabiting couple: Non-dependent children_2021,Lone parent: dependent children_2021,Lone parent: non-dependent children_2021,Other with dependent children_2021,All other types_2021,Number of sales 2011_weighted,Number of sales 2021_weighted
0,E01000001,E09000001,City of London,876,355,178,3,33,8,247,...,13,99,13,0,10,7,1,61,141.0,123.0
1,E01000002,E09000001,City of London,830,314,213,8,44,4,206,...,10,96,10,3,13,15,2,64,233.0,144.0
2,E01000003,E09000001,City of London,817,184,143,1,239,56,166,...,15,104,16,5,22,21,1,73,168.0,95.0
3,E01000005,E09000001,City of London,467,24,22,0,133,179,92,...,16,28,12,4,29,25,20,47,41.0,10.0
4,E01032739,E09000001,City of London,676,104,88,1,5,6,358,...,10,148,1,0,7,8,4,154,250.0,107.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4989,E01035718,E09000033,Westminster,996,380,122,4,7,37,328,...,34,27,4,3,23,29,41,49,189.0,113.0
4990,E01035719,E09000033,Westminster,567,142,100,8,95,26,161,...,18,48,12,1,38,25,12,67,75.0,41.0
4991,E01035720,E09000033,Westminster,558,78,91,14,107,104,134,...,10,52,23,1,33,54,5,74,84.0,46.0
4992,E01035721,E09000033,Westminster,1482,195,117,20,279,454,365,...,30,49,8,4,64,59,27,81,100.0,89.0


In [60]:
print(df_census.columns.to_list())

['LSOA code', 'local authority code', 'local authority name', 'All Households_2011', 'Owned outright_2011', 'Owned with a mortgage or loan_2011', 'Shared ownership _2011', 'Rented from Local Authority_2011', 'Other social rented_2011', 'Private landlord or letting agency_2011', 'Other private rented_2011', 'Rent free_2011', 'All Households_2021', 'Owned outright_2021', 'Owned with a mortgage or loan_2021', 'Shared ownership _2021', 'Rented from Local Authority_2021', 'Other social rented_2021', 'Private landlord or letting agency_2021', 'Other private rented_2021', 'Rent free_2021', 'All usual residents_2011', 'White British_2011', 'White Irish_2011', 'White Gypsy/Irish Traveller_2011', 'White Other_2011', 'Mixed White and Asian_2011', 'Mixed White and Black African_2011', 'Mixed White and Black Caribbean_2011', 'Mixed Other_2011', 'Asian Bangladeshi_2011', 'Asian Chinese_2011', 'Asian Indian_2011', 'Asian Pakistani_2011', 'Asian Other_2011', 'Black African_2011', 'Black Caribbean_2011

In [61]:
df_census['pct_owner_occupied_2011'] = (df_census['Owned outright_2011'] + 
                                        df_census['Owned with a mortgage or loan_2011']) / df_census['All Households_2011']
df_census['pct_owner_occupied_2021'] = (df_census['Owned outright_2021'] + 
                                        df_census['Owned with a mortgage or loan_2021']) / df_census['All Households_2021']
df_census['delta_owner_occupied'] = df_census['pct_owner_occupied_2021'] - df_census['pct_owner_occupied_2011']

In [62]:
df_census['pct_private_rented_2011'] = (df_census['Private landlord or letting agency_2011'] + 
                                        df_census['Other private rented_2011']) / df_census['All Households_2011']
df_census['pct_private_rented_2021'] = (df_census['Private landlord or letting agency_2021'] + 
                                        df_census['Other private rented_2021']) / df_census['All Households_2021']
df_census['delta_private_rented'] = df_census['pct_private_rented_2021'] - df_census['pct_private_rented_2011']

In [63]:
df_census['pct_social_rented_2011'] = (df_census['Rented from Local Authority_2011'] + 
                                       df_census['Other social rented_2011']) / df_census['All Households_2011']
df_census['pct_social_rented_2021'] = (df_census['Rented from Local Authority_2021'] + 
                                       df_census['Other social rented_2021']) / df_census['All Households_2021']
df_census['delta_social_rented'] = df_census['pct_social_rented_2021'] - df_census['pct_social_rented_2011']

In [64]:
df_census['pct_professional_2011'] = (df_census['1. Managers, directors and senior officials_2011'] + 
                                      df_census['2. Professional occupations_2011']) / df_census['All usual residents aged 16-74 in employment_2011']
df_census['pct_professional_2021'] = (df_census['1. Managers, directors and senior officials_2021'] + df_census['2. Professional occupations_2021']) / df_census['All usual residents aged 16 and over in employment_2021']
df_census['delta_professional'] = df_census['pct_professional_2021'] - df_census['pct_professional_2011']

In [65]:
df_census['age_25_34_2011'] = (df_census['Aged 25 to 29_2011'] + 
                               df_census['Aged 30 to 34_2011']) / df_census['All usual residents_2011']
df_census['age_25_34_2021'] = (df_census['Aged 25 to 29_2021'] + 
                               df_census['Aged 30 to 34_2021']) / df_census['All usual residents_2021']
df_census['delta_age_25_34'] = df_census['age_25_34_2021'] - df_census['age_25_34_2011']

In [66]:
df_census['pct_minority_ethnic_2011'] = (df_census['Mixed White and Asian_2011'] + 
                                         df_census['Mixed White and Black African_2011'] + 
                                         df_census['Mixed White and Black Caribbean_2011'] + 
                                         df_census['Mixed Other_2011'] + 
                                         df_census['Asian Bangladeshi_2011'] + 
                                         df_census['Asian Chinese_2011'] + 
                                         df_census['Asian Indian_2011'] + 
                                         df_census['Asian Pakistani_2011'] + 
                                         df_census['Asian Other_2011'] + 
                                         df_census['Black African_2011'] + 
                                         df_census['Black Caribbean_2011'] + 
                                         df_census['Black Other_2011'] + 
                                         df_census['Other Arab_2011'] + 
                                         df_census['Other Any other_2011']) / df_census['All usual residents_2011']
df_census['pct_minority_ethnic_2021'] = (df_census['Mixed White and Asian_2021'] + 
                                         df_census['Mixed White and Black African_2021'] + 
                                         df_census['Mixed White and Black Caribbean_2021'] + 
                                         df_census['Mixed Other_2021'] + 
                                         df_census['Asian Bangladeshi_2021'] + 
                                         df_census['Asian Chinese_2021'] + 
                                         df_census['Asian Indian_2021'] + 
                                         df_census['Asian Pakistani_2021'] + 
                                         df_census['Asian Other_2021'] + 
                                         df_census['Black African_2021'] + 
                                         df_census['Black Caribbean_2021'] + 
                                         df_census['Black Other_2021'] + 
                                         df_census['Other Arab_2021'] + 
                                         df_census['Other Any other_2021']) / df_census['All usual residents_2021']
df_census['delta_minority_ethnic'] = df_census['pct_minority_ethnic_2021'] - df_census['pct_minority_ethnic_2011'] 

In [67]:
df_census['pct_families_with_children_2021'] = (df_census['Married or civil partnership couple: Dependent children_2021'] + 
                                                df_census['Lone parent: dependent children_2021'] + 
                                                df_census['Cohabiting couple: Dependent children_2021'] + 
                                                df_census['Other with dependent children_2021']) / df_census['All Households_2021']

In [68]:
df_census['delta_property_sales'] = df_census['Number of sales 2021_weighted'] - df_census['Number of sales 2011_weighted']

In [69]:
df_census = df_census[['LSOA code', 'local authority code', 'local authority name', 
                'delta_age_25_34', 'delta_property_sales', 'delta_minority_ethnic',
                'pct_families_with_children_2021',
                'pct_professional_2011', 'pct_professional_2021', 'delta_professional', 
                'pct_owner_occupied_2011', 'pct_owner_occupied_2021', 'delta_owner_occupied',
                'delta_social_rented', 'pct_social_rented_2011', 'pct_social_rented_2021',  
                'pct_private_rented_2011', 'pct_private_rented_2021', 'delta_private_rented']]

In [70]:
df_census

,LSOA code,local authority code,local authority name,delta_age_25_34,delta_property_sales,delta_minority_ethnic,pct_families_with_children_2021,pct_professional_2011,pct_professional_2021,delta_professional,pct_owner_occupied_2011,pct_owner_occupied_2021,delta_owner_occupied,delta_social_rented,pct_social_rented_2011,pct_social_rented_2021,pct_private_rented_2011,pct_private_rented_2021,delta_private_rented
0,E01000001,E09000001,City of London,-0.000400,-18.0,0.042607,0.109134,0.661987,0.691244,0.029257,0.608447,0.571767,-0.036680,-0.021893,0.046804,0.024911,0.301370,0.398577,0.097207
1,E01000002,E09000001,City of London,0.074702,-89.0,0.103227,0.083636,0.646154,0.722286,0.076132,0.634940,0.523636,-0.111303,-0.022680,0.057831,0.035152,0.263855,0.438788,0.174932
2,E01000003,E09000001,City of London,0.012247,-73.0,0.042510,0.081773,0.489529,0.577844,0.088316,0.400245,0.371429,-0.028816,-0.064525,0.361077,0.296552,0.216646,0.330049,0.113403
3,E01000005,E09000001,City of London,-0.011236,-31.0,0.121495,0.221992,0.273109,0.366397,0.093288,0.098501,0.085062,-0.013439,0.024852,0.668094,0.692946,0.216274,0.219917,0.003643
4,E01032739,E09000001,City of London,-0.026440,-143.0,0.102410,0.044218,0.568987,0.669764,0.100777,0.284024,0.244898,-0.039126,-0.003801,0.016272,0.012472,0.572485,0.739229,0.166744
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4989,E01035718,E09000033,Westminster,-0.032790,-76.0,0.014648,0.210027,0.492778,0.571274,0.078496,0.504016,0.455285,-0.048732,-0.019786,0.044177,0.024390,0.364458,0.513550,0.149092
4990,E01035719,E09000033,Westminster,0.002265,-34.0,0.157911,0.207278,0.511986,0.586765,0.074778,0.426808,0.343354,-0.083453,0.049254,0.213404,0.262658,0.301587,0.386076,0.084489
4991,E01035720,E09000033,Westminster,0.005297,-38.0,0.077457,0.193162,0.520064,0.485968,-0.034097,0.302867,0.251282,-0.051585,0.038958,0.378136,0.417094,0.268817,0.300855,0.032037
4992,E01035721,E09000033,Westminster,-0.028051,-11.0,0.116808,0.143731,0.371912,0.492606,0.120694,0.210526,0.202599,-0.007927,-0.011421,0.494602,0.483180,0.265182,0.305046,0.039864


In [ ]:
df_census.to_csv("../Output/data for gentrification classification_.csv", index=False)